# DDL: `dbspend360_cluster_spends`

Creates the per-cluster spend table for **shared / interactive clusters** (`cluster_source IN ('UI','API')`).
Sibling to `dbspend360_total_job_spends` (which is JOB-scoped) so job-table semantics stay clean.

Populated by `dbspend360_cluster_spends_app`.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

**Notes**
- `cluster_name`, `owned_by`, `data_security_mode` are denormalized at ETL time so historical rows
  survive later config changes on `system.compute.clusters` (e.g. ownership transfer).
- Merge key: `(cluster_id, workspace_id, usage_date)`. No `job_id` / `run_id` because shared cluster
  DBU rows have both NULL.

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_cluster_spends (
  cluster_id          STRING,
  workspace_id        STRING,
  cluster_source      STRING,
  cluster_name        STRING,
  owned_by            STRING,
  data_security_mode  STRING,
  usage_date          DATE,
  cloud_cost          DOUBLE,
  compute_cost        DOUBLE,
  storage_cost        DOUBLE,
  network_cost        DOUBLE,
  other_cost          DOUBLE,
  databricks_cost     DOUBLE,
  currency            STRING,
  total_cost          DOUBLE,
  created_at          TIMESTAMP,
  updated_at          TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_cluster_spends")